# LANL 3SBB — model vs experiment, one case at a time

For every experimental case the notebook renders

1. a 3D view of the building with the damage location highlighted, and
2. a 3×3 grid of the 9 accelerometer FRFs comparing the experimental
   median (blue) to the reduced-order model (red dashed) for that case.

Cases without a model counterpart (e.g. Mass-only configurations) are
shown with the experimental curve only; the model panel is left blank.


## 0. Colab bootstrap

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO   = 'https://github.com/grcarmenaty/PhD_LANL.git'
BRANCH = 'claude/fix-amplitudes-sensor-comparison-nzoFm'
WORK   = Path('/content/PhD_LANL')

if 'google.colab' in sys.modules:
    if not WORK.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                        REPO, str(WORK)], check=True)
    os.chdir(WORK)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'h5py', 'numpy', 'scipy', 'matplotlib'], check=False)
print('cwd:', os.getcwd())
for f in ('median_frfs.h5', 'synthetic_frfs.h5'):
    p = Path(f)
    print(f'  {f}: {"OK" if p.exists() else "MISSING"}')


## 1. Imports and data

In [ ]:
import re
import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from pathlib import Path

matplotlib.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 9,
})

EXP_H5 = Path('median_frfs.h5')
SYN_H5 = Path('synthetic_frfs.h5')


In [ ]:
# ── Experimental medians ───────────────────────────────────────────────────
with h5py.File(EXP_H5, 'r') as f:
    exp_freq    = f['freq'][:]
    median_frf  = f['median_frf'][:]                       # (61, 1601, 9)
    exp_cnames  = [c.decode() for c in f['case_names'][:]]
    exp_counts  = f['case_counts'][:]
exp_idx_of = {n: i for i, n in enumerate(exp_cnames)}
print(f'Experimental: {len(exp_cnames)} cases, {median_frf.shape}, '
      f'{exp_freq[0]:.2f}-{exp_freq[-1]:.1f} Hz')


In [ ]:
# ── Synthetic FRFs (model output) ──────────────────────────────────────────
with h5py.File(SYN_H5, 'r') as f:
    raw        = f['frfs'][:]
    syn_freqs  = f['freqs'][:]
    syn_cnames = [n.decode() if isinstance(n, bytes) else str(n)
                  for n in f['case_names'][:]]
    units      = str(f.attrs.get('units', '(m/s^2)/N')).lower().replace(' ', '')

# Auto-rescale based on the file's own units attribute (no more blind /1000)
if 'mm' in units:    scale = 1e-3
elif 'cm' in units:  scale = 1e-2
else:                scale = 1.0
syn_data = raw * scale
print(f'Synthetic: {len(syn_cnames)} model cases, units={units}, scale={scale:g}')


## 2. Damage parser and exp ↔ model name matcher

In [ ]:
# Channel layout (HDF5 column order)
SIG_NAMES = [2, 5, 6, 7, 8, 11, 12, 13, 14]
CH_TO_FLOOR = {
    0: 'base',  4: 'base',  8: 'base',
    3: 'fl1',   7: 'fl1',
    2: 'fl2',   6: 'fl2',
    1: 'fl3',   5: 'fl3',
}

def parse_damage(name: str):
    """Return a structured spec describing the damage in *name*.

    The spec is a list of tuples (kind, storey, **info) so that combined
    damages such as 'D(85%) 1BD + D(85%) 2BD + Mass Base' can be rendered
    with multiple highlights on a single 3D plot.

    Storeys are 0-based (storey 0 = base→floor1, storey 1 = floor1→floor2,
    storey 2 = floor2→floor3).  'plate' index runs 0..3 with 0=base plate.
    """
    s = name.lower()
    if 'pristine' in s:
        return [('pristine',)]
    out = []

    # Bolt damage (D(X%) nBD / Damage (X%) nBD / nAD)
    pct_iter = re.finditer(r'(?:d|damage)\s*\(?\s*(\d+)\s*%?\s*\)?', s)
    pcts = [int(m.group(1)) for m in pct_iter]
    bd = re.findall(r'(\d)\s*bd', s)
    ad = re.findall(r'(\d)\s*ad', s)
    if pcts and (bd or ad):
        all_st = bd + ad
        if len(pcts) == 1:
            pcts = pcts * len(all_st)
        for i, st in enumerate(all_st):
            kind = 'bolt_below' if i < len(bd) else 'bolt_above'
            out.append((kind, int(st) - 1, {'pct': pcts[min(i, len(pcts)-1)]}))

    # Cracks
    for m in re.finditer(r'crack\s*(\d+)\s*mm.*?(\d)\s*bd|crack\s*(\d)\s*bd\s*(\d+)\s*mm', s):
        if m.group(1):
            size = int(m.group(1)); st = int(m.group(2))
        else:
            st = int(m.group(3)); size = int(m.group(4))
        out.append(('crack', st - 1, {'size_mm': size}))

    # Holes
    for m in re.finditer(r'hole\s*(\d+)\s*mm.*?(\d)\s*[ab]d', s):
        out.append(('hole', int(m.group(2)) - 1, {'size_mm': int(m.group(1))}))

    # Added masses
    if 'mass' in s:
        if 'base'         in s: out.append(('mass', 0, {}))
        if 'first floor'  in s or 'mass 1f' in s: out.append(('mass', 1, {}))
        if 'second floor' in s: out.append(('mass', 2, {}))
        if 'third floor'  in s: out.append(('mass', 3, {}))

    return out or [('unknown',)]


def normalize_for_match(name: str):
    """Build a signature used for exp ↔ synthetic matching.

    Mass-only cases yield ('mass_only',) and won't match any synthetic
    case (the model does not yet capture added masses).  Combined
    damage+mass cases match the underlying damage signature.
    """
    full = parse_damage(name)
    if full == [('pristine',)]:
        return ('pristine',)
    non_mass = [t for t in full if t[0] != 'mass']
    if not non_mass:
        return ('mass_only',)
    spec = non_mass
    pcts = tuple(sorted(t[2]['pct'] for t in spec
                        if t[0].startswith('bolt') and t[2].get('pct')))
    sts  = tuple(sorted(t[1] for t in spec if t[0] != 'unknown'))
    cracks = tuple(sorted(t[2]['size_mm'] for t in spec if t[0] == 'crack'))
    holes  = tuple(sorted(t[2]['size_mm'] for t in spec if t[0] == 'hole'))
    return ('dmg', pcts, sts, cracks, holes)


# Build map from signature → synthetic case index
syn_sig_to_idx = {}
for i, n in enumerate(syn_cnames):
    sig = normalize_for_match(n)
    syn_sig_to_idx.setdefault(sig, i)


def synthetic_match(case_name):
    """Return synthetic-case index that best matches *case_name*, or None."""
    return syn_sig_to_idx.get(normalize_for_match(case_name))


# Sanity check
matched = sum(1 for n in exp_cnames if synthetic_match(n) is not None)
print(f'{matched} / {len(exp_cnames)} experimental cases have a model counterpart')
print('Examples:')
for n in ['Pristine', 'Mass Base', 'Hole 4mm 1BD',
          'D(85%) 1BD + D(85%) 2BD + Mass Base']:
    if n in exp_cnames:
        si = synthetic_match(n)
        tag = syn_cnames[si] if si is not None else '(no model)'
        print(f'  {n!r:55s} -> {tag}')

## 3. 3D rendering of the building with damage indicators

In [ ]:
# Geometry constants (same as model_3sbb.py)
PL  = 0.305         # plate side length         (m)
PT  = 0.0254        # plate thickness           (m)
CLX = 0.0254        # column wide dimension     (m)
CLY = 0.0064        # column thin dimension     (m)
CGAP = 0.0005       # column-to-plate gap       (m)
SH  = PT + 0.1524   # storey height             (m)

z_pl = [k * SH + PT/2 for k in range(4)]   # 4 plate centre z's

xl = CLX / 2
xh = PL - xl
cx = PL / 2

# 9 sensor positions in HDF5 channel order
SENSOR_POS = [
    (xl, PL, z_pl[0]),  # S2  base +Y xl  (inverted polarity in exp)
    (xl, PL, z_pl[3]),  # S5  fl3  +Y xl
    (xl, PL, z_pl[2]),  # S6  fl2  +Y xl
    (xl, PL, z_pl[1]),  # S7  fl1  +Y xl
    (xh, PL, z_pl[0]),  # S8  base +Y xh
    (xh, PL, z_pl[3]),  # S11 fl3  +Y xh
    (xh, PL, z_pl[2]),  # S12 fl2  +Y xh
    (xh, PL, z_pl[1]),  # S13 fl1  +Y xh
    (cx, 0., z_pl[0]),  # S14 base -Y cx
]
SHAKER_POS = (cx, 0., z_pl[0])

# Column attachment points (4 corners) in plate xy plane
COL_XY = [
    (xl, -CLY/2 - CGAP),
    (xh, -CLY/2 - CGAP),
    (xl, PL + CLY/2 + CGAP),
    (xh, PL + CLY/2 + CGAP),
]


def _box_faces(x0, x1, y0, y1, z0, z1):
    v = [(x0,y0,z0),(x1,y0,z0),(x1,y1,z0),(x0,y1,z0),
         (x0,y0,z1),(x1,y0,z1),(x1,y1,z1),(x0,y1,z1)]
    return [[v[i] for i in idx]
            for idx in [(0,1,2,3),(4,5,6,7),(0,1,5,4),(3,2,6,7),(0,3,7,4),(1,2,6,5)]]


def render_building(ax, case_name):
    """Render the 3SBB on *ax* (must be a 3D axes) with damage indicators."""
    spec = parse_damage(case_name)

    # Highlight maps: storey index → colour for column highlight
    storey_colour = {}
    plate_mass    = set()
    column_marks  = []     # list of (storey, kind, info)

    for tag in spec:
        if tag[0] == 'pristine':
            pass
        elif tag[0].startswith('bolt'):
            st = tag[1]
            pct = tag[2].get('pct', 0)
            storey_colour[st] = ('bolt', pct)
        elif tag[0] in ('crack', 'hole'):
            column_marks.append(tag)
            storey_colour.setdefault(tag[1], (tag[0], tag[2].get('size_mm', 0)))
        elif tag[0] == 'mass':
            plate_mass.add(tag[1])

    # Plates
    for k, zc in enumerate(z_pl):
        zb, zt = zc - PT/2, zc + PT/2
        face_clr = 'silver'
        ax.add_collection3d(Poly3DCollection(
            _box_faces(0, PL, 0, PL, zb, zt),
            alpha=0.20, facecolor=face_clr, edgecolor='gray', lw=0.4))
        ax.text(PL/2, PL/2, zt + 0.005, ['Base','Floor 1','Floor 2','Floor 3'][k],
                ha='center', va='bottom', fontsize=7, color='dimgray')

    # Columns (4 corners x 3 storeys)
    for cxc, cyc in COL_XY:
        for st in range(3):
            col_kind = storey_colour.get(st)
            if col_kind is None:
                colour, lw = 'steelblue', 3.0
            elif col_kind[0] == 'bolt':
                # Lighter red for milder damage
                pct = col_kind[1]
                t = min(1.0, pct / 85.0)
                colour = (0.65 + 0.35*t, 0.15 - 0.10*t, 0.15)
                lw = 4.0
            elif col_kind[0] == 'crack':
                colour, lw = 'orange', 4.0
            elif col_kind[0] == 'hole':
                colour, lw = 'black', 4.0
            else:
                colour, lw = 'steelblue', 3.0
            ax.plot([cxc, cxc], [cyc, cyc],
                    [z_pl[st], z_pl[st+1]],
                    color=colour, lw=lw, solid_capstyle='round')

    # Localised crack / hole markers (small spheres at column mid-height)
    for kind, st, info in column_marks:
        zc = (z_pl[st] + z_pl[st+1]) / 2.0
        for cxc, cyc in COL_XY[:1]:        # mark on first corner column
            ax.scatter([cxc], [cyc], [zc],
                       s=120 if kind == 'hole' else 90,
                       c='black' if kind == 'hole' else 'orange',
                       marker='o' if kind == 'hole' else 's',
                       edgecolor='red', zorder=8)
            ax.text(cxc, cyc - 0.04, zc,
                    f"{kind} {info.get('size_mm','?')}mm",
                    fontsize=6, color='red')

    # Mass blocks on top of plates
    for k in plate_mass:
        zc = z_pl[k] + PT/2
        m_h = 0.022
        ax.add_collection3d(Poly3DCollection(
            _box_faces(PL/2 - 0.04, PL/2 + 0.04,
                       PL/2 - 0.04, PL/2 + 0.04,
                       zc, zc + m_h),
            alpha=0.85, facecolor='goldenrod', edgecolor='black', lw=0.6))
        ax.text(PL/2, PL/2, zc + m_h + 0.005, 'mass',
                ha='center', fontsize=6, color='darkgoldenrod')

    # Sensors
    for (sx, sy, sz), nm in zip(SENSOR_POS, SIG_NAMES):
        ax.scatter([sx], [sy], [sz], s=42, c='tab:blue',
                   marker='^', zorder=6, depthshade=False)
        ax.text(sx, sy + 0.015, sz + 0.008, f'S{nm}',
                fontsize=6, color='tab:blue')

    # Shaker
    sx, sy, sz = SHAKER_POS
    ax.scatter([sx], [sy], [sz], s=180, c='darkorange',
               marker='*', zorder=7, depthshade=False)
    ax.text(sx, sy - 0.04, sz + 0.005, 'shaker',
            ha='center', fontsize=6, color='darkorange')

    ax.set_xlim(0, PL); ax.set_ylim(-0.05, PL + 0.05)
    ax.set_zlim(0, z_pl[3] + 0.04)
    ax.set_xlabel('X', labelpad=2, fontsize=7)
    ax.set_ylabel('Y', labelpad=2, fontsize=7)
    ax.set_zlabel('Z', labelpad=2, fontsize=7)
    ax.tick_params(labelsize=6, pad=0)
    ax.view_init(elev=20, azim=-55)


## 4. Per-case overview

Each row below = one experimental case. Left: 3D model with damage
indicators. Right: 3×3 grid of |H| for all 9 sensors, experimental
median (blue) over the model (red dashed) — the model line is absent
for cases without a synthetic counterpart (e.g. Mass-only).


In [ ]:
# 3x3 sensor grid layout: rows = floor (top→bottom), cols = column position
GRID = [
    [1, 5, None],     # floor 3:  S5 (xl), S11 (xh)
    [2, 6, None],     # floor 2:  S6 (xl), S12 (xh)
    [3, 7, None],     # floor 1:  S7 (xl), S13 (xh)
    [0, 4,    8],     # base:     S2 (xl), S8 (xh), S14 (cx -Y)
]
ROW_LABEL = ['Floor 3', 'Floor 2', 'Floor 1', 'Base plate']
COL_LABEL = ['low-X (xl)', 'high-X (xh)', 'centre (cx)']


def render_case(case_name):
    si      = synthetic_match(case_name)
    H_exp   = median_frf[exp_idx_of[case_name]]            # (1601, 9)
    H_mod   = syn_data[si] if si is not None else None     # (1601, 9) or None
    n_meas  = int(exp_counts[exp_idx_of[case_name]])

    fig = plt.figure(figsize=(15, 7.5))
    gs  = GridSpec(4, 7, width_ratios=[1.4, 1.4, 1.4, 0.05, 1, 1, 1],
                   wspace=0.35, hspace=0.35)
    ax3d = fig.add_subplot(gs[:, :3], projection='3d')
    render_building(ax3d, case_name)
    title_extra = f' — model: {syn_cnames[si]}' if si is not None else ' — model: n/a'
    ax3d.set_title(f'{case_name}\n({n_meas} measurements aggregated){title_extra}',
                   fontsize=10)

    for r, row in enumerate(GRID):
        for c, ch in enumerate(row):
            ax = fig.add_subplot(gs[r, 4 + c])
            if ch is None:
                ax.set_visible(False); continue
            ax.semilogy(exp_freq, np.abs(H_exp[:, ch]),
                        color='tab:blue', lw=1.2, label='exp median')
            if H_mod is not None:
                ax.semilogy(syn_freqs, np.abs(H_mod[:, ch]),
                            'r--', lw=1.0, label='model')
            ax.set_xlim(0, 100); ax.set_ylim(1e-3, 1e1)
            ax.grid(True, which='both', ls=':', alpha=0.3)
            ax.tick_params(labelsize=6)
            ax.set_title(f'S{SIG_NAMES[ch]} — {ROW_LABEL[r]}',
                         fontsize=8, pad=2)
            if r == 3: ax.set_xlabel('Hz', fontsize=7)
            if c == 0: ax.set_ylabel('|H| (m/s²)/N', fontsize=7)
            if r == 0 and c == 0:
                ax.legend(fontsize=6, loc='lower right')
    plt.show()


# Iterate over every experimental case.  To restrict to a subset, edit the list.
CASES_TO_PLOT = list(exp_cnames)

print(f'Rendering {len(CASES_TO_PLOT)} cases ...\n')
for nm in CASES_TO_PLOT:
    render_case(nm)


## 5. How to make the model better

After fixing the units bug, the residual error has a clear structure:
model and experiment agree to within ~50 % at the first resonance, but
the model **over-predicts amplitude at mode 2** (~50 Hz) by ~2× and
**misses anti-resonance dips** that the experiment shows clearly.

Concrete next steps, ordered by expected impact:

1. **Per-mode damping calibration against the log-FRF.**
   The current calibration fits frequencies and pins ζ₂=ζ₃≈0.0075.
   That is way too low — exp peaks at modes 2/3 are ~3-5× lower than
   the model. Run `python calibrate_3sbb_amplitude.py` to fit ζ₁, ζ₂,
   ζ₃ against `log10|H|` on all 9 sensors. Expected: mode-2 amplitude
   error drops from 2× to <1.3×.

2. **Add rocking and torsion DOFs.**
   The reduced model is a 4-DOF Y-translation chain whose modal
   frequency ratios are bounded to `[1, 1.85, 2.41]`.  The experimental
   ratios are `[1, 2.39, 3.26]` — outside that range.  Extending each
   plate to 3 DOF (x, y, θ) and solving a 12-DOF eigenproblem unlocks
   the missing physics and explains the anti-resonances visible in
   the experimental FRFs (rocking modes producing zeros).

3. **Replace the global JSR with per-joint stiffness.**
   `joint_stiffness_ratio` is currently a single number for all 24
   bolted joints. Bolt damage is local; allowing per-joint JSR (or at
   least per-storey, per-corner) lets the model represent asymmetric
   damage cases (1AD vs 1BD, single-bolt loosening, mass-induced
   asymmetry).

4. **Damage-physics-specific submodels.**
   - *Bolt damage*: replace stiffness reduction with a Coulomb-friction
     contact whose normal load = bolt preload × torque ratio.
   - *Crack*: column → 2 segments joined by a flexibility-reduced
     section (Castigliano flexibility hinge based on crack depth).
   - *Hole*: localised section-property reduction over the hole length.
   - *Mass*: explicit added mass + rotational inertia at the plate
     attachment point (currently masses are not modelled at all).

5. **Validate against a 3D continuum FE digital twin.**
   Build a sister model in ANSYS / FEniCS / Code Aster.  Use it to
   bound modelling errors of the reduced order model and to identify
   missing physics (clamping, accelerometer mass loading, cabling).

6. **Bayesian calibration.**
   Replace point estimates with posteriors (`emcee` / `pymc`).  This is
   essential when the model is used for damage *detection*: the FRF
   change must be larger than the parameter-uncertainty FRF cone.

7. **Time-domain validation.**
   The experimental excitation was a sine sweep.  Once the FRF agrees,
   convolve the model FRF with the recorded shaker drive and compare
   to the measured time histories — exposes phase / non-minimum-phase
   effects that magnitude-only FRF fitting hides.
